In [ ]:
# Import Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score

import shap

In [ ]:
#Loading the Dataset

df = pd.read_csv("../data/telco_customer_churn.csv")

df.head()

In [ ]:
df.info()

df.describe()

df.isnull().sum()

In [ ]:
#Data Cleaning

df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

df = df.dropna()

df.drop('customerID', axis=1, inplace=True)

In [ ]:
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

In [ ]:
#Exploratory Data Analysis
#Distribution of Churn

sns.countplot(x='Churn', data=df)

plt.title("Customer Churn Distribution")

plt.show()

In [ ]:
##Churn by Contract Type

sns.countplot(x='Contract', hue='Churn', data=df)

plt.xticks(rotation=45)

plt.show()

In [ ]:
#Churn by Monthly Charges

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)

plt.show()

In [ ]:
#Feature Engineering

df = pd.get_dummies(
    df,
    drop_first=True
)

In [ ]:
#Train-Test Split

X = df.drop('Churn', axis=1)

y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Model Experimentation

To identify the best predictive model for customer churn, we compare multiple machine learning algorithms including Logistic Regression and Random Forest.

In [ ]:
#Modeling - Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=2000)

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("Logistic Regression Results\n")

print(classification_report(y_test, pred_lr))

In [ ]:
#Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("Random Forest Results\n")

print(classification_report(y_test, pred_rf))

In [ ]:
#model evaluation

from sklearn.metrics import roc_auc_score

prob_lr = lr.predict_proba(X_test)[:,1]
prob_rf = rf.predict_proba(X_test)[:,1]

print("Logistic Regression ROC AUC:", roc_auc_score(y_test, prob_lr))
print("Random Forest ROC AUC:", roc_auc_score(y_test, prob_rf))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [100,150,200],
    "max_depth": [None,10,20],
    "min_samples_split": [2,5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

In [ ]:
best_rf = grid.best_estimator_

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

pred_best = best_rf.predict(X_test)

print("Tuned Random Forest Results\n")

print(classification_report(y_test, pred_best))

prob_best = best_rf.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test, prob_best))

In [ ]:
#Feature Importance

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

importance.sort_values().tail(10).plot(kind="barh")

plt.show()

In [ ]:
#Explainability with SHAP

import shap
import matplotlib.pyplot as plt

explainer = shap.Explainer(best_rf)

shap_values = explainer(X_test)

#beeswarm plot
shap.plots.beeswarm(shap_values[:,:,1], show=False)

# Save figure
plt.savefig("../outputs/shap_summary.png",dpi=300, bbox_inches="tight")

plt.close()





In [ ]:
#bar plot
shap.plots.bar(shap_values[:,:,1], show=False)

plt.savefig("../outputs/shap_bar.png",dpi=300, bbox_inches="tight")

plt.close()


## Key Insights

• Month-to-month contract customers show significantly higher churn risk.

• Customers with higher monthly charges are more likely to churn.

• Customers with longer tenure demonstrate stronger retention.

• Contract type and tenure are major drivers of churn behavior.